## 02 Pair Generation and Labeling

This notebook creates the pairwise review dataset for curriculum-overlap analysis.

Its purpose is straightforward:

**generate all unique course pairs and prepare the gold review file for manual human labeling.**

Notebook 02 treats Notebook 01 as the source of truth.  
It does not rebuild course text fields. Instead, it uses the cleaned outputs already prepared there.

Main tasks:
- load and validate the cleaned course dataset
- generate all unique unordered course pairs
- attach human-readable review text for both sides of each pair
- create stable `pair_id` values
- save the final review dataset for manual annotation

Main output:
- `data/gold_pairs_review.csv`

Important methodological choices:
- the notebook generates the full pair universe
- it does not create dev/test splits
- it uses `review_text_full` for human review
- leakage-safe course-level splitting is handled later in Notebook 03

In [77]:
# Cell 01 - Import libraries and define project paths

from itertools import combinations
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

CLEAN_CSV_PATH = DATA_DIR / "courses_clean.csv"
GOLD_REVIEW_CSV_PATH = DATA_DIR / "gold_pairs_review.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", DATA_DIR.resolve())
print("Clean dataset path:", CLEAN_CSV_PATH.resolve())
print("Gold review output path:", GOLD_REVIEW_CSV_PATH.resolve())
print("Clean dataset exists:", CLEAN_CSV_PATH.exists())

Project root: C:\Users\user\Downloads\curriculum-overlap-poc
Data directory: C:\Users\user\Downloads\curriculum-overlap-poc\data
Clean dataset path: C:\Users\user\Downloads\curriculum-overlap-poc\data\courses_clean.csv
Gold review output path: C:\Users\user\Downloads\curriculum-overlap-poc\data\gold_pairs_review.csv
Clean dataset exists: True


### 1. Load and validate the cleaned dataset

Notebook 01 has already produced the cleaned course dataset used here.

This section checks:
- that the cleaned file exists
- that the expected columns are present
- that the review text needed for manual labeling is available
- that course identifiers remain unique

The goal is to fail early if the upstream dataset is incomplete or inconsistent.

In [78]:
# Cell 02 - Load cleaned course dataset

if not CLEAN_CSV_PATH.exists():
    raise FileNotFoundError(f"Clean dataset not found: {CLEAN_CSV_PATH.resolve()}")

courses_df = pd.read_csv(CLEAN_CSV_PATH, encoding="utf-8", keep_default_na=False)

print("courses_df shape:", courses_df.shape)
print("Columns:")
print(courses_df.columns.tolist())

print("\nOne transposed preview row:")
display(courses_df.head(1).T)

courses_df shape: (50, 20)
Columns:
['degree_programme', 'course_name', 'course_unit_code', 'credits', 'teaching_language_text', 'responsible_person_text', 'objective_text', 'competences_knowledge_text', 'competences_practice_text', 'content_text', 'prerequisites_text', 'assessment_grading_scale', 'assessment_pass_fail_criteria', 'course_name_normalized', 'prerequisites_text_normalized', 'embedding_text_compact', 'embedding_text_main', 'embedding_text_with_prerequisites', 'review_text_compact', 'review_text_full']

One transposed preview row:


,0
degree_programme,Information and Communication Technology
course_name,Data Networks
course_unit_code,TT00CD70
credits,5
teaching_language_text,"Finnish, English"
responsible_person_text,Pasi Hyytiäinen
objective_text,You learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data network. You are ...
competences_knowledge_text,You know the main modeling methods of data networks. You have knowledge and understanding of data networks and their effects. You know the basic principles ...
competences_practice_text,You can apply data network skills to communication between devices.
content_text,Principles of network structure and protocol design. Principles of Internet communication. OSI model layers. Design and implementation of local area network...


In [79]:
# Cell 03 - Validate required columns and key input quality

required_columns = [
    "course_name",
    "course_unit_code",
    "review_text_full",
]

missing_required_columns = [col for col in required_columns if col not in courses_df.columns]
if missing_required_columns:
    raise ValueError(f"Missing required columns in cleaned dataset: {missing_required_columns}")

duplicate_course_codes = courses_df["course_unit_code"].duplicated().sum()
duplicate_course_names = courses_df["course_name"].duplicated().sum()
empty_review_text_full = (courses_df["review_text_full"].fillna("").str.strip() == "").sum()

print("Missing required columns:", missing_required_columns)
print("Total rows:", len(courses_df))
print("Unique course codes:", courses_df["course_unit_code"].nunique())
print("Unique course names:", courses_df["course_name"].nunique())
print("Duplicate course codes:", duplicate_course_codes)
print("Duplicate course names:", duplicate_course_names)
print("Empty review_text_full rows:", empty_review_text_full)

if duplicate_course_codes > 0:
    raise ValueError("Duplicate course_unit_code values found in cleaned dataset.")

if duplicate_course_names > 0:
    raise ValueError("Duplicate course_name values found in cleaned dataset.")

if empty_review_text_full > 0:
    raise ValueError("Some courses have empty review_text_full values. Notebook 02 expects these to be complete.")

Missing required columns: []
Total rows: 50
Unique course codes: 50
Unique course names: 50
Duplicate course codes: 0
Duplicate course names: 0
Empty review_text_full rows: 0


### 2. Manual overlap decision rubric

The goal of manual review is to assign a **binary gold label** for whether a pair shows meaningful curricular overlap.

#### Label as `1` (meaningful overlap) when
The two courses substantially overlap in one or more of the following:
- central learning objectives
- main content themes
- core technical concepts
- practical skills or competencies taught

A positive label should reflect overlap that would matter in curriculum analysis, not just loose topical similarity.

#### Label as `0` (no meaningful overlap) when
The courses only share:
- the same broad field or department
- a few general keywords
- minor tool overlap
- a prerequisite relationship without substantial content overlap
- surface-level similarity without clearly overlapping teaching focus

#### Borderline cases
If a pair is difficult to judge:
- choose the label that best reflects the **dominant teaching overlap**
- mark `review_status` as `reviewed`
- use `review_notes` to explain the uncertainty briefly

#### Important annotation principle
The question is not whether the courses are related.  
The question is whether they overlap enough in taught content or competencies to be considered meaningfully overlapping for curriculum-overlap detection.

### 3. Generate all unique unordered course pairs

For 50 courses, the full unordered pair universe contains:

\[
\frac{50 \times 49}{2} = 1225
\]

Pairs are generated without self-pairs and without reversed duplicates.

Each pair receives a stable `pair_id` based on sorted course codes.  
This guarantees that the identifier is deterministic and suitable for later benchmarking and evaluation.

In [80]:
# Cell 04 - Select source columns needed for pair generation

pair_source_df = courses_df[[
    "course_unit_code",
    "course_name",
    "review_text_full",
]].copy()

print("pair_source_df shape:", pair_source_df.shape)
display(pair_source_df.head(3))

pair_source_df shape: (50, 3)


,course_unit_code,course_name,review_text_full
0,TT00CD70,Data Networks,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...
1,TT00CD71,Linux Basics,"Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera..."
2,TT00CD72,Servers and containers,Course code: TT00CD72\nCourse name: Servers and containers\nObjective: You understand the fundamental types and functions of servers. You understand the key...


In [81]:
# Cell 05 - Generate all unique unordered course pairs

pair_source_df = pair_source_df.sort_values("course_unit_code").reset_index(drop=True)

course_index_pairs = list(combinations(pair_source_df.index, 2))
expected_pair_count = len(pair_source_df) * (len(pair_source_df) - 1) // 2

print("Expected pair count:", expected_pair_count)
print("Actual generated index-pair count:", len(course_index_pairs))
print("First 5 index pairs:", course_index_pairs[:5])

if len(course_index_pairs) != expected_pair_count:
    raise ValueError("Generated pair count does not match expected count.")

Expected pair count: 1225
Actual generated index-pair count: 1225
First 5 index pairs: [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5)]


In [82]:
# Cell 06 - Build the pair review dataset

pair_records = []

for idx1, idx2 in course_index_pairs:
    course_1 = pair_source_df.loc[idx1]
    course_2 = pair_source_df.loc[idx2]

    code_1 = str(course_1["course_unit_code"]).strip()
    code_2 = str(course_2["course_unit_code"]).strip()

    pair_id = f"{code_1}__{code_2}"

    pair_records.append({
        "pair_id": pair_id,
        "course_unit_code_1": code_1,
        "course_name_1": course_1["course_name"],
        "review_text_1": course_1["review_text_full"],
        "course_unit_code_2": code_2,
        "course_name_2": course_2["course_name"],
        "review_text_2": course_2["review_text_full"],
        "overlap_label": "",
        "review_status": "",
        "review_notes": "",
    })

pairs_df = pd.DataFrame(pair_records)

print("pairs_df shape:", pairs_df.shape)
display(pairs_df.head(3))

pairs_df shape: (1225, 10)


,pair_id,course_unit_code_1,course_name_1,review_text_1,course_unit_code_2,course_name_2,review_text_2,overlap_label,review_status,review_notes
0,TT00CD70__TT00CD71,TT00CD70,Data Networks,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...,TT00CD71,Linux Basics,"Course code: TT00CD71\nCourse name: Linux Basics\nObjective: After completing the course, you will understand the most important concepts of the Linux opera...",,,
1,TT00CD70__TT00CD72,TT00CD70,Data Networks,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...,TT00CD72,Servers and containers,Course code: TT00CD72\nCourse name: Servers and containers\nObjective: You understand the fundamental types and functions of servers. You understand the key...,,,
2,TT00CD70__TT00CD73,TT00CD70,Data Networks,Course code: TT00CD70\nCourse name: Data Networks\nObjective: You learn the structure and protocols of data networks used by computers and other end devices...,TT00CD73,Digital Technology,Course code: TT00CD73\nCourse name: Digital Technology\nObjective: You understand the basics of digital number systems and know how to apply the knowledge i...,,,


In [83]:
# Cell 07 - Validate pair dataset integrity

expected_pair_count = len(pair_source_df) * (len(pair_source_df) - 1) // 2

integrity_summary = {
    "rows_in_pairs_df": len(pairs_df),
    "expected_pair_count": expected_pair_count,
    "unique_pair_id_count": pairs_df["pair_id"].nunique(),
    "duplicate_pair_id_count": pairs_df["pair_id"].duplicated().sum(),
    "self_pairs_count": (pairs_df["course_unit_code_1"] == pairs_df["course_unit_code_2"]).sum(),
    "empty_review_text_1_count": (pairs_df["review_text_1"].fillna("").str.strip() == "").sum(),
    "empty_review_text_2_count": (pairs_df["review_text_2"].fillna("").str.strip() == "").sum(),
    "reversed_or_unsorted_pair_count": (pairs_df["course_unit_code_1"] >= pairs_df["course_unit_code_2"]).sum(),
}

display(pd.Series(integrity_summary))

if len(pairs_df) != expected_pair_count:
    raise ValueError("pairs_df row count does not match expected pair count.")

if pairs_df["pair_id"].duplicated().any():
    raise ValueError("Duplicate pair_id values found.")

if (pairs_df["course_unit_code_1"] == pairs_df["course_unit_code_2"]).any():
    raise ValueError("Self-pairs found, which should never happen.")

if (pairs_df["review_text_1"].fillna("").str.strip() == "").any():
    raise ValueError("Some review_text_1 values are empty.")

if (pairs_df["review_text_2"].fillna("").str.strip() == "").any():
    raise ValueError("Some review_text_2 values are empty.")

if (pairs_df["course_unit_code_1"] >= pairs_df["course_unit_code_2"]).any():
    raise ValueError("Pair ordering is not strictly deterministic by ascending course_unit_code.")

rows_in_pairs_df                   1225
expected_pair_count                1225
unique_pair_id_count               1225
duplicate_pair_id_count               0
self_pairs_count                      0
empty_review_text_1_count             0
empty_review_text_2_count             0
reversed_or_unsorted_pair_count       0
dtype: int64

### 4. Assemble the final review dataset

The final output of this notebook is the gold review file used for manual labeling.

The saved file contains:
- pair identifiers
- both course codes and course names
- both human-readable review texts
- blank labeling fields ready for manual review

In [84]:
# Cell 08 - Assemble final gold review dataset

gold_review_df = pairs_df[[
    "pair_id",
    "course_unit_code_1",
    "course_name_1",
    "review_text_1",
    "course_unit_code_2",
    "course_name_2",
    "review_text_2",
    "overlap_label",
    "review_status",
    "review_notes",
]].copy()

# Save in deterministic order for reproducibility and clean version control diffs
gold_review_df = gold_review_df.sort_values("pair_id").reset_index(drop=True)

# Keep review text spreadsheet-friendly while preserving visible section boundaries
gold_review_df["review_text_1"] = gold_review_df["review_text_1"].str.replace("\n", " | ", regex=False)
gold_review_df["review_text_2"] = gold_review_df["review_text_2"].str.replace("\n", " | ", regex=False)

print("gold_review_df shape:", gold_review_df.shape)
display(gold_review_df.head(10))

gold_review_df shape: (1225, 10)


,pair_id,course_unit_code_1,course_name_1,review_text_1,course_unit_code_2,course_name_2,review_text_2,overlap_label,review_status,review_notes
0,TT00CD70__TT00CD71,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD71,Linux Basics,"Course code: TT00CD71 | Course name: Linux Basics | Objective: After completing the course, you will understand the most important concepts of the Linux ope...",,,
1,TT00CD70__TT00CD72,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD72,Servers and containers,Course code: TT00CD72 | Course name: Servers and containers | Objective: You understand the fundamental types and functions of servers. You understand the k...,,,
2,TT00CD70__TT00CD73,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD73,Digital Technology,Course code: TT00CD73 | Course name: Digital Technology | Objective: You understand the basics of digital number systems and know how to apply the knowledge...,,,
3,TT00CD70__TT00CD74,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD74,Information Systems and Architecture,"Course code: TT00CD74 | Course name: Information Systems and Architecture | Objective: After completing the course, you understand what the concepts softwar...",,,
4,TT00CD70__TT00CD75,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD75,Windows Basics,Course code: TT00CD75 | Course name: Windows Basics | Objective: You understand and know how to use Windows management tools and resource monitoring tools. ...,,,
5,TT00CD70__TT00CD76,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD76,Scripting and Automatization,Course code: TT00CD76 | Course name: Scripting and Automatization | Objective: You understand the benefits of automation related to the management of operat...,,,
6,TT00CD70__TT00CD77,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD77,Basics of Programming,Course code: TT00CD77 | Course name: Basics of Programming | Objective: You know the most essential concepts of programming and understand the basic theory....,,,
7,TT00CD70__TT00CD78,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD78,Frontend Programming,Course code: TT00CD78 | Course name: Frontend Programming | Objective: You know and understand the possibilities of different application frameworks to impl...,,,
8,TT00CD70__TT00CD79,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD79,Basics of Web Development,"Course code: TT00CD79 | Course name: Basics of Web Development | Objective: In the course, you familiarise yourself with the basics of creating websites fol...",,,
9,TT00CD70__TT00CD80,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD80,JavaScript Programming,Course code: TT00CD80 | C

### 5. Diagnostics before saving

At this stage:
- total pair count should be 1225
- all label fields should still be blank
- all review status fields should still be blank

After manual review begins, the same file will reflect your progress.

In [85]:
# Cell 09 - Final integrity checks before saving

expected_pair_count = len(courses_df) * (len(courses_df) - 1) // 2

print("Expected pair count:", expected_pair_count)
print("Actual pair count:", len(gold_review_df))

print("\nUnique pair_id checks:")
print("gold_review_df:", gold_review_df["pair_id"].nunique())

if len(gold_review_df) != expected_pair_count:
    raise ValueError("Gold review pair count does not match expected full pair count.")

if gold_review_df["pair_id"].nunique() != len(gold_review_df):
    raise ValueError("gold_review_df contains duplicate pair_id values.")

Expected pair count: 1225
Actual pair count: 1225

Unique pair_id checks:
gold_review_df: 1225


In [86]:
# Cell 10 - Blank-label summary before manual review

gold_review_df["overlap_label"] = gold_review_df["overlap_label"].fillna("").astype(str).str.strip()
gold_review_df["review_status"] = gold_review_df["review_status"].fillna("").astype(str).str.strip()

label_filled_mask = gold_review_df["overlap_label"] != ""
reviewed_mask = gold_review_df["review_status"] != ""

summary_df = pd.DataFrame([
    {"metric": "total_pairs", "value": len(gold_review_df)},
    {"metric": "labeled_pairs", "value": int(label_filled_mask.sum())},
    {"metric": "unlabeled_pairs", "value": int((~label_filled_mask).sum())},
    {"metric": "reviewed_pairs", "value": int(reviewed_mask.sum())},
    {"metric": "unreviewed_pairs", "value": int((~reviewed_mask).sum())},
])

display(summary_df)

,metric,value
0,total_pairs,1225
1,labeled_pairs,0
2,unlabeled_pairs,1225
3,reviewed_pairs,0
4,unreviewed_pairs,1225


In [87]:
# Cell 11 - Save Notebook 02 output safely

if GOLD_REVIEW_CSV_PATH.exists():
    raise FileExistsError(
        "gold_pairs_review.csv already exists.\n"
        "To protect manual labeling work, this notebook will not overwrite it.\n"
        "Rename, move, or delete the existing file first if you intentionally want to regenerate it."
    )

gold_review_df.to_csv(GOLD_REVIEW_CSV_PATH, index=False, encoding="utf-8")

print("Saved gold review dataset to:", GOLD_REVIEW_CSV_PATH.resolve())

Saved gold review dataset to: C:\Users\user\Downloads\curriculum-overlap-poc\data\gold_pairs_review.csv


In [88]:
# Cell 12 - Preview the final output

print("Gold review dataset preview:")
display(gold_review_df.head(5))

Gold review dataset preview:


,pair_id,course_unit_code_1,course_name_1,review_text_1,course_unit_code_2,course_name_2,review_text_2,overlap_label,review_status,review_notes
0,TT00CD70__TT00CD71,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD71,Linux Basics,"Course code: TT00CD71 | Course name: Linux Basics | Objective: After completing the course, you will understand the most important concepts of the Linux ope...",,,
1,TT00CD70__TT00CD72,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD72,Servers and containers,Course code: TT00CD72 | Course name: Servers and containers | Objective: You understand the fundamental types and functions of servers. You understand the k...,,,
2,TT00CD70__TT00CD73,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD73,Digital Technology,Course code: TT00CD73 | Course name: Digital Technology | Objective: You understand the basics of digital number systems and know how to apply the knowledge...,,,
3,TT00CD70__TT00CD74,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD74,Information Systems and Architecture,"Course code: TT00CD74 | Course name: Information Systems and Architecture | Objective: After completing the course, you understand what the concepts softwar...",,,
4,TT00CD70__TT00CD75,TT00CD70,Data Networks,Course code: TT00CD70 | Course name: Data Networks | Objective: You learn the structure and protocols of data networks used by computers and other end devic...,TT00CD75,Windows Basics,Course code: TT00CD75 | Course name: Windows Basics | Objective: You understand and know how to use Windows management tools and resource monitoring tools. ...,,,


### Manual review instructions

The file produced by this notebook is:

- **`gold_pairs_review.csv`**

This file should now be reviewed manually.

#### How to review each row

For each pair:
- read `review_text_1`
- read `review_text_2`
- decide whether the courses have **meaningful curricular overlap**

#### Fill these columns

- **overlap_label**
  - `1` = meaningful overlap
  - `0` = no meaningful overlap
  - blank = not reviewed yet

- **review_status**
  - `reviewed` if you have checked the pair
  - blank if not reviewed yet

- **review_notes**
  - optional short explanation
  - recommended for borderline or surprising cases

#### Decision rule

Label a pair as **`1`** when the courses substantially overlap in:
- central learning objectives
- main course content
- core competencies or technical skills taught

Do **not** label a pair as `1` only because:
- they belong to the same broad subject area
- they share a few general keywords
- they mention similar tools in passing
- one is a prerequisite for the other without substantial content overlap

#### Borderline cases

If the decision is not obvious:
- choose the label that best reflects the dominant teaching overlap
- keep `review_status` as `reviewed`
- use `review_notes` to record the reason briefly

#### Important limitation to remember

This gold-label process is expert-informed but still based on a single reviewer, so some subjectivity is unavoidable. The purpose of `review_notes` is to make difficult cases more transparent and easier to revisit later if needed.

#### What happens next

After manual review is complete:
- `gold_pairs_review.csv` becomes the ground-truth label file
- Notebook 03 will use:
  - `courses_clean.csv` from Notebook 01 for course text representations
  - `gold_pairs_review.csv` from Notebook 02 for human labels

Notebook 03 should begin only after the manual review is complete.

## Final note

This notebook creates the full pair universe for the JAMK curriculum-overlap benchmark and prepares it for consistent manual labeling.

### What this notebook does
- loads the cleaned course dataset from Notebook 01
- generates all unique unordered course pairs
- assigns stable and deterministic `pair_id` values
- uses `review_text_full` as the default human-readable review surface
- saves the main review dataset for manual labeling and later benchmarking

### What this notebook intentionally does not do
- it does not rebuild course text fields
- it does not perform dev/test splitting
- it does not compute embedding benchmarks
- it does not filter out pairs before manual review

### Why this design is appropriate
The corpus is small enough that the full pair universe is manageable, and realistic curriculum overlap is expected to be rare. For that reason, it is methodologically stronger to preserve the complete pair population and handle class imbalance honestly rather than artificially balancing or filtering the data too early.

The notebook also aligns the human review process with Notebook 01 by using labeled review text for people and leaving the actual modeling work to Notebook 03 after the gold labels are complete.

This makes the pair-labeling workflow transparent, reproducible, and suitable for both thesis evaluation and practical proof-of-concept use.